In [1]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import os

# Paths
DATA_DAILY = Path("data/daily")
DATA_RUNTIME = Path("data/runtime")
DATA_AVERAGES = Path("data/averages")
OUTPUT_RUNTIME = Path("output/runtime")


for p in (DATA_DAILY, DATA_RUNTIME, DATA_AVERAGES, OUTPUT_RUNTIME):
    p.mkdir(parents=True, exist_ok=True)

In [2]:

# Load latest daily CSV
daily_file = sorted(DATA_DAILY.glob("hot_stocks_*.csv"))[-1]
df_daily = pd.read_csv(daily_file)

df_daily.shape

(40, 12)

In [3]:

# Load averages
df_avg = pd.read_csv(DATA_AVERAGES / "average_hot_scores.csv")

# Merge for comparison
df_merge = df_daily.merge(df_avg[['symbol','HotScore']], on='symbol', how='left', suffixes=('_today','_avg'))

# Compute RuntimeScore (example)
df_merge['RuntimeScore'] = 0.5*df_merge['HotScore_today'] + 0.5*(df_merge['HotScore_today']/df_merge['HotScore_avg'])

# Filter top suggestions
df_runtime = df_merge[df_merge['RuntimeScore'] > df_merge['RuntimeScore'].quantile(0.7)].sort_values('RuntimeScore', ascending=False)
df_runtime.shape

(12, 14)

In [4]:
# Save CSV with timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
csv_file = DATA_RUNTIME / f"runtime_suggestions_{timestamp}.csv"
df_runtime.to_csv(csv_file, index=False)

In [5]:
import plotly.express as px

fig = px.bar(   
    df_runtime.sort_values(by=["regularMarketPrice"], ascending=True).head(40), 
    x="RuntimeScore",
    y="symbol", 
    color="HotScore_today",
    text="regularMarketPrice",
    hover_data=["TrendScore"], 
    title="Runtime Scores vs Volatility",
    color_continuous_scale="ice",
)

fig.update_traces(textposition="inside", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 20 Runtime Score Stocks",
    xaxis_title="Runtime Score",
    yaxis_title="STOCKS",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1000,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

chart_path = os.path.join(OUTPUT_RUNTIME, "bar_runtime.html")
fig.write_html(chart_path, include_plotlyjs="cdn")
 